# Improvement Ablation Benchmark

Measures the impact of each pipeline improvement on **read-level** and **site-level** performance.

## Configurations Compared

| # | Config | Fix A | Fix B | HMM |
|---|--------|-------|-------|-----|
| 1 | V2 only | - | - | off |
| 2 | kNN only | - | - | off |
| 3 | kNN+HMM (old default) | no | no | unsup 2-state |
| 4 | kNN+HMM (Fix A) | **yes** | no | unsup 2-state |
| 5 | wkNN+HMM (Fix A+B) | **yes** | **yes** | unsup 2-state |
| 6 | wkNN only (Fix B) | - | **yes** | off |

- **Fix A**: Consistent short-trajectory fallback (kNN instead of V2)
- **Fix B**: Distance-weighted kNN scoring (closer neighbors count more)

## 1. Setup

In [ ]:
import sys
from pathlib import Path

if Path.cwd().name == 'notebooks':
    sys.path.insert(0, str(Path.cwd().parent))

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
from sklearn.metrics import (
    roc_curve, auc, precision_recall_curve,
    average_precision_score, roc_auc_score,
)

from baleen.eventalign import (
    load_results,
    compute_sequential_modification_probabilities,
    aggregate_all,
)

plt.rcParams.update({'figure.dpi': 120, 'font.size': 10, 'axes.titlesize': 11})

## 2. File Paths

In [ ]:
BALEEN_PKL = Path('../output/pipeline_results.pkl')
POSITION_OFFSET = 3

for candidate in [BALEEN_PKL,
                  Path('../results/pipeline_results.pkl'),
                  Path('../output/pipeline_results.pkl'),
                  Path('output/pipeline_results.pkl')]:
    if candidate.exists():
        BALEEN_PKL = candidate
        break

print(f'Using: {BALEEN_PKL}  (exists={BALEEN_PKL.exists()})')

## 3. Ground Truth: E. coli rRNA

In [ ]:
KNOWN_MODIFICATIONS = {
    ('ecoli16S', 516):  ('\u03a8',     'pseudouridine'),
    ('ecoli23S', 746):  ('\u03a8',     'pseudouridine'),
    ('ecoli23S', 955):  ('\u03a8',     'pseudouridine'),
    ('ecoli23S', 1911): ('\u03a8',     'pseudouridine'),
    ('ecoli23S', 1917): ('\u03a8',     'pseudouridine'),
    ('ecoli23S', 2457): ('\u03a8',     'pseudouridine'),
    ('ecoli23S', 2504): ('\u03a8',     'pseudouridine'),
    ('ecoli23S', 2580): ('\u03a8',     'pseudouridine'),
    ('ecoli23S', 2604): ('\u03a8',     'pseudouridine'),
    ('ecoli23S', 2605): ('\u03a8',     'pseudouridine'),
    ('ecoli16S', 966):  ('m2G',   'N2-methylguanosine'),
    ('ecoli16S', 1207): ('m2G',   'N2-methylguanosine'),
    ('ecoli16S', 1516): ('m2G',   'N2-methylguanosine'),
    ('ecoli23S', 1835): ('m2G',   'N2-methylguanosine'),
    ('ecoli23S', 2445): ('m2G',   'N2-methylguanosine'),
    ('ecoli16S', 967):  ('m5C',   '5-methylcytidine'),
    ('ecoli16S', 1407): ('m5C',   '5-methylcytidine'),
    ('ecoli23S', 1962): ('m5C',   '5-methylcytidine'),
    ('ecoli23S', 747):  ('m5U',   '5-methyluridine'),
    ('ecoli23S', 1939): ('m5U',   '5-methyluridine'),
    ('ecoli16S', 1518): ('m6,6A', 'N6,N6-dimethyladenosine'),
    ('ecoli16S', 1519): ('m6,6A', 'N6,N6-dimethyladenosine'),
    ('ecoli23S', 1618): ('m6A',   'N6-methyladenosine'),
    ('ecoli23S', 2030): ('m6A',   'N6-methyladenosine'),
    ('ecoli16S', 527):  ('m7G',   'N7-methylguanosine'),
    ('ecoli23S', 2069): ('m7G',   'N7-methylguanosine'),
    ('ecoli23S', 2498): ('Cm',    \"2'-O-methylcytosine\"),
    ('ecoli23S', 2449): ('D',     'dihydrouridine'),
    ('ecoli23S', 2251): ('Gm',    \"2'-O-methylguanosine\"),
    ('ecoli23S', 2552): ('Um',    \"2'-O-methyluridine\"),
    ('ecoli23S', 2501): ('ho5C',  '5-hydroxycytidine'),
    ('ecoli23S', 745):  ('m1G',   '1-methylguanosine'),
    ('ecoli23S', 2503): ('m2A',   '2-methyladenosine'),
    ('ecoli23S', 1915): ('m3\u03a8',   '3-methylpseudouridine'),
    ('ecoli16S', 1498): ('m3U',   '3-methyluridine'),
    ('ecoli16S', 1402): ('m4Cm',  \"N4,2'-O-dimethylcytidine\"),
}

KNOWN_MOD_PIPELINE = {
    (c, p - POSITION_OFFSET): v
    for (c, p), v in KNOWN_MODIFICATIONS.items()
}
KNOWN_MOD_SET = set(KNOWN_MOD_PIPELINE.keys())

mod_counts = Counter(v[0] for v in KNOWN_MODIFICATIONS.values())
print(f'Total known modification sites: {len(KNOWN_MODIFICATIONS)}')
for mt, count in mod_counts.most_common():
    full = next(v[1] for v in KNOWN_MODIFICATIONS.values() if v[0] == mt)
    print(f'  {mt:8s}  {count} sites   ({full})')

## 4. Load DTW Results

In [ ]:
contig_results, metadata = load_results(BALEEN_PKL)

total_pos = sum(len(cr.positions) for cr in contig_results.values())
print(f'Loaded {len(contig_results)} contigs, {total_pos} positions')
for name, cr in contig_results.items():
    n_known = sum(1 for p in cr.positions if (name, p) in KNOWN_MOD_SET)
    print(f'  {name}: {len(cr.positions)} positions, {n_known} known-modified')

## 5. Run Pipeline Configurations

Each configuration re-runs V1+V2+kNN+HMM on the **same** DTW distance matrices.

In [ ]:
def run_config(label, **kwargs):
    print(f'  {label} ...')
    out = {}
    for contig, cr in contig_results.items():
        out[contig] = compute_sequential_modification_probabilities(cr, **kwargs)
    return out

print('Running 6 configurations:')

cfg = {}

# 1. V2 only
cfg['V2 only'] = run_config('V2 only',
    run_hmm=False, emission_source='p_mod_raw', consistent_fallback=False)

# 2. kNN only
cfg['kNN only'] = run_config('kNN only',
    run_hmm=False, emission_source='p_mod_knn', consistent_fallback=False)

# 3. kNN+HMM OLD default (V2 fallback for short trajectories)
cfg['kNN+HMM (old)'] = run_config('kNN+HMM (old)',
    emission_source='p_mod_knn', consistent_fallback=False)

# 4. kNN+HMM Fix A (consistent kNN fallback)
cfg['kNN+HMM (Fix A)'] = run_config('kNN+HMM (Fix A)',
    emission_source='p_mod_knn', consistent_fallback=True)

# 5. weighted-kNN + HMM (Fix A+B)
cfg['wkNN+HMM (A+B)'] = run_config('wkNN+HMM (A+B)',
    emission_source='p_mod_knn', consistent_fallback=True, knn_weighted=True)

# 6. weighted-kNN only (Fix B, no HMM)
cfg['wkNN only (B)'] = run_config('wkNN only (B)',
    run_hmm=False, emission_source='p_mod_knn', consistent_fallback=False,
    knn_weighted=True)

print('Done.')

## 6. Extract Per-Read Scores

In [ ]:
def extract_read_scores(config_results, score_field='p_mod_hmm'):
    records = []
    for contig, cr in contig_results.items():
        cmr = config_results.get(contig)
        if cmr is None:
            continue
        for pos, pr in cr.positions.items():
            ps = cmr.position_stats.get(pos)
            if ps is None:
                continue
            is_mod = (contig, pos) in KNOWN_MOD_SET
            mod_info = KNOWN_MOD_PIPELINE.get((contig, pos), ('unmod', 'unmodified'))
            scores = getattr(ps, score_field)
            for i, name in enumerate(pr.native_read_names):
                s = float(scores[i])
                if np.isnan(s): continue
                records.append(dict(contig=contig, position=pos, read_name=name,
                    is_native=True, y_true=1 if is_mod else 0,
                    mod_type=mod_info[0], score=s))
            for j, name in enumerate(pr.ivt_read_names):
                s = float(scores[pr.n_native_reads + j])
                if np.isnan(s): continue
                records.append(dict(contig=contig, position=pos, read_name=name,
                    is_native=False, y_true=0, mod_type='unmod', score=s))
    return pd.DataFrame(records)

dfs = {label: extract_read_scores(results) for label, results in cfg.items()}

for label, df in dfs.items():
    n_pos = df['y_true'].sum()
    print(f'{label:25s}  {len(df):>7,} reads  ({n_pos:,} positive)')

## 7. Read-Level: ROC and PR Curves (Native Only)

Native reads only — the harder, more realistic evaluation.

In [ ]:
COLORS = {
    'V2 only':           '#9E9E9E',
    'kNN only':          '#90CAF9',
    'kNN+HMM (old)':     '#FFA726',
    'kNN+HMM (Fix A)':   '#66BB6A',
    'wkNN+HMM (A+B)':    '#EF5350',
    'wkNN only (B)':     '#AB47BC',
}

fig, (ax_roc, ax_pr) = plt.subplots(1, 2, figsize=(14, 5.5))
summary_rows = []

for label, df in dfs.items():
    df_nat = df[df['is_native']]
    y, s = df_nat['y_true'].values, df_nat['score'].values
    valid = ~np.isnan(s)
    if valid.sum() == 0 or len(np.unique(y[valid])) < 2: continue
    y_v, s_v = y[valid], s[valid]
    color = COLORS.get(label, '#000')

    fpr, tpr, _ = roc_curve(y_v, s_v)
    roc_val = auc(fpr, tpr)
    ax_roc.plot(fpr, tpr, label=f'{label} ({roc_val:.4f})', color=color, lw=2)

    prec, rec, _ = precision_recall_curve(y_v, s_v)
    pr_val = average_precision_score(y_v, s_v)
    ax_pr.plot(rec, prec, label=f'{label} ({pr_val:.4f})', color=color, lw=2)

    summary_rows.append(dict(Config=label, AUROC=roc_val, AUPRC=pr_val,
                             n_native=int(valid.sum()), n_pos=int(y_v.sum())))

ax_roc.plot([0,1],[0,1],'k--',alpha=0.3)
ax_roc.set(xlabel='FPR', ylabel='TPR', title='Read-Level ROC (Native Only)')
ax_roc.legend(fontsize=8, title='AUROC')
ax_pr.set(xlabel='Recall', ylabel='Precision', title='Read-Level PR (Native Only)')
ax_pr.legend(fontsize=8, title='AUPRC')
fig.tight_layout()
plt.show()

## 8. Read-Level: Performance by Modification Type

In [ ]:
mod_types = sorted({v[0] for v in KNOWN_MOD_PIPELINE.values()})
mod_auc_rows = []
for mt in mod_types:
    for label, df in dfs.items():
        df_nat = df[df['is_native']]
        mask = (df_nat['mod_type'] == mt) | (df_nat['mod_type'] == 'unmod')
        subset = df_nat[mask]
        y = (subset['mod_type'] == mt).astype(int).values
        s = subset['score'].values
        valid = ~np.isnan(s)
        if valid.sum() < 10 or len(np.unique(y[valid])) < 2: continue
        mod_auc_rows.append(dict(Mod=mt, Config=label,
            AUROC=roc_auc_score(y[valid], s[valid]),
            AUPRC=average_precision_score(y[valid], s[valid]),
            n_pos=int(y[valid].sum())))

if mod_auc_rows:
    df_mod = pd.DataFrame(mod_auc_rows)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, max(4, len(mod_types)*0.55)))
    for ax, metric in [(ax1, 'AUROC'), (ax2, 'AUPRC')]:
        pivot = df_mod.pivot(index='Mod', columns='Config', values=metric)
        pivot = pivot[[c for c in cfg.keys() if c in pivot.columns]]
        pivot.plot.barh(ax=ax, color=[COLORS.get(c,'#999') for c in pivot.columns])
        ax.set(xlabel=metric, title=f'Read-Level {metric} by Mod Type')
        ax.legend(fontsize=7)
    fig.tight_layout()
    plt.show()

## 9. Site-Level: Aggregation and FDR

In [ ]:
def site_metrics(config_results, label):
    sites = aggregate_all(config_results)
    if not sites: return {}, [], pd.DataFrame()
    rows = []
    for s in sites:
        is_mod = (s.contig, s.position) in KNOWN_MOD_SET
        rows.append(dict(contig=s.contig, position=s.position,
            mod_ratio=s.mod_ratio, pvalue=s.pvalue, padj=s.padj,
            effect_size=s.effect_size, y_true=1 if is_mod else 0))
    df_s = pd.DataFrame(rows)
    y, s = df_s['y_true'].values, df_s['mod_ratio'].values
    auroc = roc_auc_score(y, s) if len(np.unique(y)) >= 2 else float('nan')
    auprc = average_precision_score(y, s) if len(np.unique(y)) >= 2 else float('nan')
    auc_row = dict(Config=label, Site_AUROC=auroc, Site_AUPRC=auprc,
                   n_sites=len(df_s), n_mod=int(df_s['y_true'].sum()))
    fdr_rows = []
    for fdr in [0.01, 0.05, 0.1]:
        called = df_s[df_s['padj'] < fdr]
        tp = called['y_true'].sum()
        fp = len(called) - tp
        fn = df_s['y_true'].sum() - tp
        prec = tp/(tp+fp) if (tp+fp) > 0 else 0.0
        rec = tp/(tp+fn) if (tp+fn) > 0 else 0.0
        f1 = 2*prec*rec/(prec+rec) if (prec+rec) > 0 else 0.0
        fdr_rows.append(dict(Config=label, FDR=fdr, TP=tp, FP=fp, FN=fn,
                             Prec=prec, Rec=rec, F1=f1))
    return auc_row, fdr_rows, df_s

site_auc, site_fdr, site_dfs = [], [], {}
for label, results in cfg.items():
    a, f, df_s = site_metrics(results, label)
    site_auc.append(a); site_fdr.extend(f); site_dfs[label] = df_s

print('=== Site-Level AUC ===')
print(pd.DataFrame(site_auc).to_string(index=False, float_format='{:.4f}'.format))
print()
print('=== Site-Level FDR Performance ===')
print(pd.DataFrame(site_fdr).to_string(index=False, float_format='{:.4f}'.format))

## 10. Site-Level: ROC and PR Curves

In [ ]:
fig, (ax_roc, ax_pr) = plt.subplots(1, 2, figsize=(14, 5.5))
for label, df_s in site_dfs.items():
    y, s = df_s['y_true'].values, df_s['mod_ratio'].values
    valid = ~np.isnan(s)
    if valid.sum() == 0 or len(np.unique(y[valid])) < 2: continue
    color = COLORS.get(label, '#000')
    fpr, tpr, _ = roc_curve(y[valid], s[valid])
    ax_roc.plot(fpr, tpr, label=f'{label} ({auc(fpr,tpr):.4f})', color=color, lw=2)
    prec, rec, _ = precision_recall_curve(y[valid], s[valid])
    ax_pr.plot(rec, prec, label=f'{label} ({average_precision_score(y[valid],s[valid]):.4f})', color=color, lw=2)

ax_roc.plot([0,1],[0,1],'k--',alpha=0.3)
ax_roc.set(xlabel='FPR', ylabel='TPR', title='Site-Level ROC')
ax_roc.legend(fontsize=8)
ax_pr.set(xlabel='Recall', ylabel='Precision', title='Site-Level PR')
ax_pr.legend(fontsize=8)
fig.tight_layout()
plt.show()

## 11. Score Distributions: Modified vs Unmodified

In [ ]:
n_cfg = len(dfs)
fig, axes = plt.subplots(1, n_cfg, figsize=(4*n_cfg, 3.5), sharey=True)
if n_cfg == 1: axes = [axes]
for ax, (label, df) in zip(axes, dfs.items()):
    df_nat = df[df['is_native']]
    bins = np.linspace(0, 1, 40)
    ax.hist(df_nat[df_nat['y_true']==0]['score'], bins=bins, alpha=0.6,
            label='Unmod', color='#42A5F5', density=True)
    ax.hist(df_nat[df_nat['y_true']==1]['score'], bins=bins, alpha=0.6,
            label='Mod', color='#EF5350', density=True)
    ax.set(xlabel='P(mod)', title=label)
    ax.legend(fontsize=7)
axes[0].set_ylabel('Density')
fig.suptitle('Native Read Score Distributions', y=1.02)
fig.tight_layout()
plt.show()

## 12. Short vs Long Trajectory Analysis

In [ ]:
ref = cfg['kNN+HMM (Fix A)']
traj_lens = []
for cmr in ref.values():
    for traj in list(cmr.native_trajectories) + list(cmr.ivt_trajectories):
        traj_lens.append(len(traj.positions))
traj_lens = np.array(traj_lens)
n_short = (traj_lens < 3).sum()
print(f'Total trajectories: {len(traj_lens):,}')
print(f'Short (<3 pos, use fallback): {n_short:,} ({100*n_short/len(traj_lens):.1f}%)')
print(f'Long (>=3 pos, use HMM): {len(traj_lens)-n_short:,}')
print(f'Median length: {np.median(traj_lens):.0f}, mean: {traj_lens.mean():.1f}')

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.hist(traj_lens, bins=range(0, min(int(traj_lens.max())+2, 50)),
        color='#42A5F5', edgecolor='white', alpha=0.8)
ax.axvline(3, color='red', ls='--', label='HMM min_positions=3')
ax.set(xlabel='Trajectory Length', ylabel='Count', title='Read Trajectory Lengths')
ax.legend()
fig.tight_layout()
plt.show()

## 13. Delta Summary

In [ ]:
if len(summary_rows) >= 2:
    bl = summary_rows[0]
    print(f"{'Config':30s} {'AUROC':>8s} {'AUPRC':>8s}  {'dAUROC':>8s} {'dAUPRC':>8s}")
    print('-' * 70)
    for r in summary_rows:
        dr = r['AUROC'] - bl['AUROC']
        dp = r['AUPRC'] - bl['AUPRC']
        print(f"{r['Config']:30s} {r['AUROC']:8.4f} {r['AUPRC']:8.4f}"
              f"  {'+' if dr>=0 else ''}{dr:7.4f} {'+' if dp>=0 else ''}{dp:7.4f}")

if site_auc:
    print()
    bl = site_auc[0]
    print(f"{'Config':30s} {'Site AUROC':>11s} {'Site AUPRC':>11s}  {'dAUROC':>8s} {'dAUPRC':>8s}")
    print('-' * 78)
    for r in site_auc:
        dr = r['Site_AUROC'] - bl['Site_AUROC']
        dp = r['Site_AUPRC'] - bl['Site_AUPRC']
        print(f"{r['Config']:30s} {r['Site_AUROC']:11.4f} {r['Site_AUPRC']:11.4f}"
              f"  {'+' if dr>=0 else ''}{dr:7.4f} {'+' if dp>=0 else ''}{dp:7.4f}")